# الفصل الحادي والعشرون: مسار الأتمتة — استخراج البيانات من الويب (Web Scraping)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/chapter-21.ipynb)

> **الوقت المتوقع:** ~11 دقيقة قراءة · **15** قسمًا · **25** مثالًا برمجيًا.

**كيف تستخدم هذا الدفتر؟**

1. شغّل الخلايا بالترتيب من الأعلى إلى الأسفل (`Shift` + `Enter`).
2. خلايا الشيفرة قابلة للتعديل: غيّر القيم وأعد التشغيل لتجرّب.
3. خلايا 🧾 معروضة للقراءة فقط (أوامر طرفية، أو إدخال يدوي، أو مقتطف ناقص)، وخلايا ⚠️ تُظهر خطأً مقصودًا يُلتقط تلقائيًا فلا يتوقف «تشغيل الكل».

> **Kaggle:** نزّل الملف من المستودع ثم `File ▸ Import Notebook`، أو استورده مباشرة من GitHub بعد ربط حسابك.

<details>
<summary><b>محتويات الفصل</b></summary>

- **21.1** لماذا الأتمتة؟ ولماذا Web Scraping تحديدًا؟
- **21.2** قبل أن نبدأ: الأخلاقيات والقانونية في Web Scraping
- **21.3** أساسيات HTML التي تحتاجها (مراجعة سريعة)
- **21.4** مكتبة `requests`: طلب الصفحات
- **21.5** مكتبة BeautifulSoup: تحليل واستخراج البيانات
- **21.6** نمط العمل الكامل: من الطلب إلى البيانات المنظّمة
- **21.7** التعامل مع التصفّح متعدد الصفحات (Pagination)
- **21.8** حفظ البيانات المُستخرجة
- **21.9** معالجة الأخطاء بصلابة (Robust Scraping)
- **21.10** ماذا عن المواقع التي تعتمد على JavaScript؟
- **21.11** مشروع تطبيقي متكامل: مستخرج بيانات كتب شامل
- **21.12** تمارين
- **21.13** اختبر فهمك (Quiz)
- **21.14** ملخص الفصل (Cheat Sheet)
- **21.15** ماذا بعد؟

</details>

In [ ]:
#@title ⚙️ تهيئة البيئة  { display-mode: "form" }
import sys, os, random, pathlib

print("Python:", sys.version.split()[0], "|", sys.platform)
assert sys.version_info >= (3, 10), "هذا الكتاب يفترض Python 3.10 أو أحدث"

SEED = 42
random.seed(SEED)

# تثبيت الحزم الناقصة فقط (سريع إن كانت مثبّتة أصلًا)
import importlib.util as _iu
_need = [m for m in ["requests", "bs4", "lxml"] if _iu.find_spec(m) is None]
if _need:
    %pip install -q requests beautifulsoup4 lxml

# مجلد عمل معزول حتى لا تتناثر الملفات في جذر الدفتر
WORK = pathlib.Path("book_runtime")
WORK.mkdir(exist_ok=True)
os.chdir(WORK)
print("مجلد العمل:", pathlib.Path.cwd().name)

# إعداد خاص بهذا الفصل
# مهلة افتراضية لكل طلب حتى لا تعلّق الخلية إن تعذّر الوصول للشبكة
import requests as _rq
_orig_request = _rq.Session.request
def _request(self, method, url, **kw):
    kw.setdefault('timeout', 10)
    return _orig_request(self, method, url, **kw)
_rq.Session.request = _request

# asyncio.run() لا تعمل داخل Jupyter: نشغّل الحلقة في خيط منفصل
import asyncio as _asyncio, threading as _threading
def _run_in_thread(coro):
    box = {}
    def _target():
        loop = _asyncio.new_event_loop()
        try:
            box['value'] = loop.run_until_complete(coro)
        except BaseException as exc:
            box['error'] = exc
        finally:
            loop.close()
    thread = _threading.Thread(target=_target)
    thread.start()
    thread.join()
    if 'error' in box:
        raise box['error']
    return box['value']
_asyncio.run = _run_in_thread

> ℹ️ أمثلة السحب تحتاج إنترنت؛ ويمكنك تجربتها على صفحة محلية عند انقطاعه.

## 21.1 لماذا الأتمتة؟ ولماذا Web Scraping تحديدًا؟

تخيّل أنك تريد مراقبة سعر منتج معيّن في متجر إلكتروني كل يوم، أو جمع كل عناوين الأخبار من موقع معيّن، أو تتبّع تقييمات منتج على مئات الصفحات. فتح المتصفح يدويًا ونسخ البيانات واحدة تلو الأخرى قد يستغرق ساعات — وهو عمل مملّ ومتكرر بالضبط من النوع الذي صُمم الحاسوب لتوفيره عنا.

**الأتمتة (Automation)** هي استخدام البرمجة لتنفيذ مهام متكررة تلقائيًا دون تدخل بشري. و**استخراج بيانات الويب (Web Scraping)** هو أحد أكثر تطبيقات الأتمتة شيوعًا وفائدة: كتابة برنامج «يزور» صفحات الويب نيابة عنك، «يقرأ» محتواها، ويستخرج منها البيانات التي تهمك بشكل منظم.

> **🔄 مقارنة: كيف يحصل الإنسان على البيانات مقابل كيف يحصل عليها السكربت**
>
> | الإنسان | سكربت Web Scraping |
> |---|---|
> | يفتح المتصفح، يقرأ الصفحة بعينه | يرسل طلب HTTP، يستقبل HTML خام |
> | ينسخ البيانات يدويًا في Excel | يحلل HTML ويستخرج البيانات تلقائيًا |
> | دقائق لكل صفحة | أجزاء من الثانية لكل صفحة |
> | يتعب بعد 50 صفحة | يمكنه معالجة 50,000 صفحة بنفس الجهد |
> | عرضة للخطأ البشري | دقيق طالما الكود صحيح |

استخدامات Web Scraping الحقيقية تشمل: مراقبة أسعار المنافسين، جمع بيانات للتدريب على نماذج الذكاء الاصطناعي، تتبّع فرص العمل الجديدة، أرشفة الأخبار، البحث الأكاديمي، ومراقبة توفر المنتجات.

في هذا الفصل، سنتعلم مكتبتين أساسيتين:

1. **`requests`**: لإرسال طلبات HTTP والحصول على محتوى الصفحات (تمامًا كما يفعل المتصفح، لكن بدون واجهة رسومية).
2. **`BeautifulSoup`**: لتحليل محتوى HTML المُستلم واستخراج البيانات المحددة منه بسهولة.

ثم سنطبّق كل هذا في **مشروع متكامل لاستخراج بيانات كتب من موقع تدريبي**.

## 21.2 قبل أن نبدأ: الأخلاقيات والقانونية في Web Scraping

هذا قسم **إلزامي القراءة** قبل أي سطر كود.

Web Scraping قوة كبيرة، ومعها مسؤولية كبيرة. إليك القواعد الذهبية:

> **⚠️ تحذير مهم جدًا: قواعد السلوك الأخلاقي في Web Scraping**
>
> 1. **افحص ملف `robots.txt`** دائمًا قبل استخراج بيانات من أي موقع. يوجد عادة على `example.com/robots.txt` ويوضّح ما يُسمح وما لا يُسمح باستخراجه آليًا.
> 2. **اقرأ شروط الاستخدام (Terms of Service)** للموقع — بعض المواقع تمنع صراحة الاستخراج الآلي.
> 3. **لا تُثقل الخادم بالطلبات**: أضف تأخيرًا (`time.sleep()`) بين الطلبات المتتالية. إرسال آلاف الطلبات في ثوانٍ قد يُعتبر هجوم حرمان من الخدمة (DoS) حتى لو لم تقصد ذلك!
> 4. **لا تستخرج بيانات شخصية** أو محمية بحقوق نشر دون إذن.
> 5. **فضّل استخدام API رسمي** إن وُجد — معظم المواقع الكبرى (Twitter/X، Reddit، GitHub...) توفّر APIs رسمية أنظف وأكثر استقرارًا من Web Scraping.
> 6. **عرّف نفسك**: أرسل `User-Agent` واضحًا يوضّح من أنت، بدلاً من انتحال متصفح حقيقي خداعًا.
>
> في كل أمثلة هذا الفصل سنستخدم موقع **books.toscrape.com**، وهو موقع **تدريبي مُصمَّم خصيصًا وبشكل قانوني وآمن** لممارسة Web Scraping دون أي قلق أخلاقي أو قانوني.

> **💡 نصيحة خبير:** تحقق دائمًا من `robots.txt` برمجيًا إن أردت الأتمتة الكاملة، عبر مكتبة Python المدمجة `urllib.robotparser`. لكن في مشاريعك الحقيقية، القراءة اليدوية لشروط الموقع كافية ومهمة.

## 21.3 أساسيات HTML التي تحتاجها (مراجعة سريعة)

لكي نستخرج بيانات من صفحة ويب، يجب أن نفهم هيكلها الأساسي. **HTML** (HyperText Markup Language) هي اللغة التي تُبنى بها صفحات الويب، وتتكوّن من **عناصر (Elements)** محاطة بـ **وسوم (Tags)**:

```html
<html>
  <body>
    <div class="product">
      <h2 class="title">لابتوب Dell</h2>
      <p class="price">2500 ريال</p>
      <a href="/product/1">عرض التفاصيل</a>
    </div>
    <div class="product">
      <h2 class="title">ماوس لاسلكي</h2>
      <p class="price">80 ريال</p>
      <a href="/product/2">عرض التفاصيل</a>
    </div>
  </body>
</html>
```

🧾 **للقراءة فقط** — محتوى ملف `html`.

مفاهيم أساسية يجب معرفتها:

| المصطلح | الوصف | مثال |
|---|---|---|
| **Tag (وسم)** | يحدد نوع العنصر | `<div>`, `<p>`, `<h2>`, `<a>` |
| **Attribute (خاصية)** | معلومات إضافية عن العنصر | `class="product"`, `href="/product/1"` |
| **class** | يُستخدم غالبًا للتنسيق، ومفيد جدًا للاستهداف في الاستخراج | `class="price"` |
| **id** | معرّف فريد لعنصر واحد في الصفحة | `id="main-header"` |
| **Nesting (التداخل)** | العناصر تحتوي عناصر أخرى بداخلها (هيكل شجري) | `<div>` يحتوي `<h2>` و`<p>` |

> **💡 نصيحة خبير:** أفضل صديق لك في Web Scraping هو **أدوات المطوّر في المتصفح** (اضغط `F12` أو كليك يمين → «فحص العنصر» / Inspect). استخدمها دائمًا لتحديد الوسم والـ class الدقيقين للبيانات التي تريد استخراجها **قبل** كتابة أي كود.

## 21.4 مكتبة `requests`: طلب الصفحات

### التثبيت

```bash
uv add requests
uv add --dev beautifulsoup4 lxml
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

(`lxml` هو محلّل HTML سريع نستخدمه مع BeautifulSoup).

### أول طلب GET

In [ ]:
import requests

response = requests.get("https://books.toscrape.com/")

print(response.status_code)   # 200 يعني نجاح (كما تعلمنا في الفصل 19)
print(response.text[:500])    # أول 500 حرف من محتوى HTML الخام

### فحص حالة الاستجابة

In [ ]:
import requests

response = requests.get("https://books.toscrape.com/")

if response.status_code == 200:
    print("✅ نجح الطلب!")
elif response.status_code == 404:
    print("❌ الصفحة غير موجودة")
else:
    print(f"⚠️ حدث خطأ: {response.status_code}")

# طريقة أنظف وأكثر احترافية
try:
    response.raise_for_status()  # يرفع استثناء تلقائيًا إذا كان status_code خطأ
    print("✅ نجح الطلب!")
except requests.HTTPError as e:
    print(f"❌ خطأ HTTP: {e}")

> **🔄 مقارنة: `status_code` اليدوي مقابل `raise_for_status()`**
>
> | الطريقة | السلوك |
> |---|---|
> | `if response.status_code == 200` | يتطلب تحققًا يدويًا من كل كود محتمل |
> | `response.raise_for_status()` | يرفع `requests.HTTPError` تلقائيًا لأي كود 4xx أو 5xx، فتتعامل معه بـ `try/except` كما تعلمنا في الفصل 11 |

### إضافة Headers (مهم جدًا)

بعض المواقع ترفض الطلبات التي لا تحمل `User-Agent` يشبه متصفحًا حقيقيًا:

In [ ]:
import requests

headers = {
    "User-Agent": "Mozilla/5.0 (Educational Scraper Bot - contact@example.com)"
}

response = requests.get("https://books.toscrape.com/", headers=headers)
print(response.status_code)

### تحديد مهلة زمنية (Timeout) — إلزامي دائمًا

In [ ]:
import requests

try:
    response = requests.get(
        "https://books.toscrape.com/",
        timeout=10,  # ثوانٍ -- إذا لم يرد الخادم خلالها، يُرفع استثناء
    )
except requests.Timeout:
    print("⏱️ انتهت المهلة الزمنية -- الخادم بطيء جدًا أو لا يستجيب")
except requests.ConnectionError:
    print("🔌 تعذّر الاتصال -- تحقق من الإنترنت أو الرابط")

> **⚠️ تحذير شائع جدًا: نسيان `timeout`**
> بدون `timeout`، إذا توقف الخادم عن الاستجابة، سيبقى برنامجك **معلّقًا إلى الأبد** (hang) دون أي رسالة خطأ. ضع `timeout` في **كل** طلب `requests` تكتبه، بلا استثناء — هذه قاعدة احترافية لا نقاش فيها.

### معاملات الاستعلام (Query Parameters) و POST

```python
import requests

# GET مع query parameters (مثل الفصل 19)
response = requests.get(
    "https://api.example.com/search",
    params={"q": "python", "page": 1},
    timeout=10,
)
# الرابط الفعلي سيكون: https://api.example.com/search?q=python&page=1

# POST لإرسال بيانات (مثلاً تسجيل الدخول أو إرسال نموذج)
response = requests.post(
    "https://api.example.com/login",
    json={"username": "ahmad", "password": "secret"},
    timeout=10,
)
```

🧾 **للقراءة فقط** — المثال يستخدم عنوانًا وهميًا (`api.example.com`) للتوضيح.

### التعامل مع استجابات JSON

كثير من المواقع الحديثة تستخدم APIs تُرجع JSON مباشرة (بدون HTML)، وهذا أسهل بكثير من استخراج HTML:

In [ ]:
import requests

response = requests.get("https://api.github.com/users/python", timeout=10)
data = response.json()  # تحويل تلقائي من JSON إلى dict/list بايثون

print(data["name"])
print(data["public_repos"])

> **💡 نصيحة خبير:** قبل أن تلجأ لتحليل HTML المعقّد، تحقّق دائمًا إن كان الموقع يوفّر **API رسمي** أو حتى API داخلي غير موثّق تستخدمه صفحة الويب نفسها (يمكنك اكتشافه عبر تبويب "Network" في أدوات المطوّر، والبحث عن طلبات تُرجع JSON). التعامل مع JSON مباشرة أسهل وأكثر استقرارًا بكثير من استخراج HTML.

### استخدام `Session` لعدة طلبات متتالية

In [ ]:
import requests

# Session يحافظ تلقائيًا على الكوكيز بين الطلبات (مفيد بعد تسجيل الدخول مثلاً)
with requests.Session() as session:
    session.headers.update({"User-Agent": "Educational Bot"})

    login_response = session.post(
        "https://example.com/login",
        data={"user": "ahmad", "pass": "1234"},
        timeout=10,
    )

    # هذا الطلب سيستخدم نفس الكوكيز تلقائيًا
    profile_response = session.get("https://example.com/profile", timeout=10)

## 21.5 مكتبة BeautifulSoup: تحليل واستخراج البيانات

بعد الحصول على محتوى HTML الخام عبر `requests`، نحتاج طريقة ذكية لاستخراج البيانات منه. هنا يأتي دور **BeautifulSoup**.

### الإعداد الأساسي

In [ ]:
import requests
from bs4 import BeautifulSoup

response = requests.get("https://books.toscrape.com/", timeout=10)
soup = BeautifulSoup(response.text, "lxml")  # "lxml" هو المحلل (parser) المستخدم

print(soup.title)        # <title>All products | Books to Scrape...</title>
print(soup.title.text)   # All products | Books to Scrape - Sandbox

> **🔄 مقارنة: المحلّلات (Parsers) المتاحة في BeautifulSoup**
>
> | Parser | السرعة | يحتاج تثبيت خارجي | ملاحظات |
> |---|---|---|---|
> | `"html.parser"` | متوسطة | لا (مدمج في Python) | خيار جيد افتراضي |
> | `"lxml"` | سريع جدًا | نعم (`uv add lxml`) | الأفضل للاستخدام الاحترافي |
> | `"html5lib"` | بطيء | نعم | الأكثر تسامحًا مع HTML غير صحيح |

### البحث عن العناصر: `find` و `find_all`

In [ ]:
# العثور على أول عنصر مطابق
first_product = soup.find("article", class_="product_pod")

# العثور على كل العناصر المطابقة -> تُرجع قائمة
all_products = soup.find_all("article", class_="product_pod")
print(f"عدد المنتجات: {len(all_products)}")

# البحث بالوسم فقط
all_headings = soup.find_all("h3")

# البحث بخاصية id
header = soup.find(id="messages")

# البحث بعدة شروط
links = soup.find_all("a", href=True)  # كل روابط <a> التي تملك خاصية href

> **⚠️ تحذير: `class` كلمة محجوزة في Python!**
> لا يمكنك كتابة `soup.find("div", class="price")` لأن `class` كلمة مفتاحية في بايثون (تُستخدم لتعريف الأصناف كما في الفصل 13). الحل: استخدم `class_` بشرطة سفلية:
> ```python
> soup.find("div", class_="price")   # ✅ صحيح
> ```

### استخراج النص والخصائص

In [ ]:
product = soup.find("article", class_="product_pod")

title_tag = product.find("h3").find("a")
title = title_tag["title"]           # الوصول لخاصية HTML (title attribute)
# أو
title = title_tag.get("title")       # أكثر أمانًا -- يُرجع None إن لم توجد بدلاً من خطأ

price_tag = product.find("p", class_="price_color")
price_text = price_tag.text          # النص الظاهر للمستخدم: "£51.77"
price_text_stripped = price_tag.get_text(strip=True)  # يزيل المسافات الزائدة تلقائيًا

> **💡 نصيحة خبير:** استخدم دائمًا `.get("attribute")` بدلاً من `["attribute"]` عند استخراج خصائص HTML، لأن `.get()` تُرجع `None` بأمان إذا لم تكن الخاصية موجودة، بينما `[]` يرفع `KeyError` ويُوقف برنامجك. هذا تطبيق مباشر لمبدأ «التعامل الدفاعي مع البيانات» الذي تعلمناه في الفصل 11.

### محددات CSS: `select` و `select_one` (الطريقة الأكثر قوة ومرونة)

إذا كنت تعرف CSS (المستخدمة لتنسيق صفحات الويب)، يمكنك استخدام نفس الصياغة للاستخراج:

In [ ]:
# محدد CSS: وسم.class
titles = soup.select("h3 a")               # كل وسم <a> داخل <h3>
prices = soup.select("p.price_color")       # كل <p> بخاصية class="price_color"
first_price = soup.select_one("p.price_color")  # أول عنصر مطابق فقط

# محددات أكثر تعقيدًا
all_product_links = soup.select("article.product_pod h3 a")

for link in all_product_links:
    print(link.get("title"), "->", link.get("href"))

> **🔄 مقارنة: `find`/`find_all` مقابل `select`/`select_one`**
>
> | الطريقة | الصياغة | متى تستخدمها |
> |---|---|---|
> | `find("div", class_="x")` | صياغة BeautifulSoup الخاصة | عند الحاجة لمرونة برمجية (شروط Python) |
> | `select("div.x")` | صياغة CSS القياسية | أسرع كتابة إذا كنت تعرف CSS، ويدعم محددات معقدة مثل `div > p:first-child` |
>
> **توصية:** تعلّم كلتا الطريقتين، لكن `select` غالبًا أسرع وأوضح للمحددات المعقدة.

### التنقل في الشجرة (Tree Navigation)

In [ ]:
product = soup.find("article", class_="product_pod")

print(product.parent)         # العنصر الأب
print(product.find_next_sibling())  # العنصر الشقيق التالي
print(product.contents)       # كل العناصر الفرعية المباشرة كقائمة

## 21.6 نمط العمل الكامل: من الطلب إلى البيانات المنظّمة

لنجمع كل شيء في مثال متكامل يستخرج بيانات كتاب واحد من صفحة تجريبية:

In [ ]:
import requests
from bs4 import BeautifulSoup

url = "https://books.toscrape.com/"
headers = {"User-Agent": "Educational Scraper (learning purposes)"}

response = requests.get(url, headers=headers, timeout=10)
response.raise_for_status()

soup = BeautifulSoup(response.text, "lxml")

books = soup.select("article.product_pod")
print(f"تم العثور على {len(books)} كتاب في الصفحة")

for book in books[:3]:  # أول 3 كتب فقط للعرض
    title = book.select_one("h3 a").get("title")
    price = book.select_one("p.price_color").get_text(strip=True)
    availability = book.select_one("p.instock.availability").get_text(strip=True)
    rating_classes = book.select_one("p.star-rating")["class"]
    rating = [c for c in rating_classes if c != "star-rating"][0]  # مثل "Three"

    print(f"📖 {title}")
    print(f"   السعر: {price}")
    print(f"   التوفر: {availability}")
    print(f"   التقييم: {rating}")
    print("-" * 40)

```
تم العثور على 20 كتاب في الصفحة
📖 A Light in the Attic
   السعر: £51.77
   التوفر: In stock
   التقييم: Three
----------------------------------------
...
```

## 21.7 التعامل مع التصفّح متعدد الصفحات (Pagination)

معظم المواقع الحقيقية توزّع البيانات على عدة صفحات. يجب أتمتة الانتقال بينها:

In [ ]:
import time

import requests
from bs4 import BeautifulSoup

BASE_URL = "https://books.toscrape.com/catalogue/page-{}.html"
headers = {"User-Agent": "Educational Scraper"}

all_titles = []
page = 1

while True:
    url = BASE_URL.format(page)
    response = requests.get(url, headers=headers, timeout=10)

    if response.status_code == 404:
        print(f"انتهت الصفحات عند الصفحة {page - 1}")
        break

    soup = BeautifulSoup(response.text, "lxml")
    books = soup.select("article.product_pod")

    for book in books:
        title = book.select_one("h3 a").get("title")
        all_titles.append(title)

    print(f"✅ تمت معالجة الصفحة {page} ({len(books)} كتاب)")
    page += 1
    time.sleep(1)  # ⚠️ احترام الخادم -- تأخير بين الطلبات إلزامي

print(f"\nإجمالي الكتب المُستخرجة: {len(all_titles)}")

> **⚠️ تحذير: لماذا `time.sleep(1)` ضرورية؟**
> بدون تأخير، سكربتك قد يُرسل عشرات أو مئات الطلبات في الثانية الواحدة — هذا:
> 1. قد يُحمّل خادم الموقع بشكل غير عادل (غير أخلاقي).
> 2. قد يؤدي لحظر عنوان IP الخاص بك تلقائيًا (Rate Limiting من جانب الموقع).
> 3. في الحالات القصوى، قد يُعتبر هجومًا إلكترونيًا من الناحية القانونية.
>
> قاعدة عامة: ثانية واحدة على الأقل بين الطلبات لموقع صغير، وأكثر من ذلك للمواقع الكبيرة حسب سياستها المعلنة في `robots.txt`.

## 21.8 حفظ البيانات المُستخرجة

البيانات المُستخرجة بلا فائدة إن بقيت في الذاكرة فقط. نحفظها كما تعلمنا في الفصل 16:

In [ ]:
import csv

data = [
    {"title": "A Light in the Attic", "price": "£51.77", "rating": "Three"},
    {"title": "Tipping the Velvet", "price": "£53.74", "rating": "One"},
]

with open("books.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["title", "price", "rating"])
    writer.writeheader()
    writer.writerows(data)

أو مباشرة عبر pandas (كما تعلمنا في الفصل 20) لتحليل فوري:

In [ ]:
import pandas as pd

df = pd.DataFrame(data)
df.to_csv("books.csv", index=False, encoding="utf-8-sig")
print(df.describe())

> **💡 نصيحة خبير:** هذا هو **الجسر الحقيقي بين المسارات**: استخرج البيانات بـ `requests`/`BeautifulSoup`، نظّمها وحلّلها بـ `pandas`، وتصوّرها بـ `matplotlib`. هذه هي خطوط الأنابيب (Pipelines) الحقيقية المستخدمة في الصناعة.

## 21.9 معالجة الأخطاء بصلابة (Robust Scraping)

البيانات الحقيقية على الويب فوضوية: صفحات مفقودة، عناصر غير موجودة أحيانًا، اتصال بطيء. سكربت احترافي يجب أن يتحمّل كل هذا دون أن ينهار:

In [ ]:
import time

import requests
from bs4 import BeautifulSoup

def fetch_page(url: str, retries: int = 3) -> requests.Response | None:
    """جلب صفحة مع إعادة محاولة عند الفشل.

    Args:
        url: الرابط المطلوب جلبه.
        retries: عدد محاولات إعادة الطلب عند الفشل.

    Returns:
        كائن Response عند النجاح، أو None عند فشل كل المحاولات.
    """
    headers = {"User-Agent": "Educational Scraper"}

    for attempt in range(1, retries + 1):
        try:
            response = requests.get(url, headers=headers, timeout=10)
            response.raise_for_status()
            return response
        except requests.Timeout:
            print(f"⏱️ محاولة {attempt}: انتهت المهلة الزمنية لـ {url}")
        except requests.HTTPError as e:
            print(f"❌ محاولة {attempt}: خطأ HTTP {e}")
        except requests.ConnectionError:
            print(f"🔌 محاولة {attempt}: تعذّر الاتصال بـ {url}")

        if attempt < retries:
            time.sleep(2 * attempt)  # تأخير متزايد (Exponential backoff مبسّط)

    print(f"🚫 فشلت كل المحاولات لـ {url}")
    return None

def extract_book_data(book_element) -> dict[str, str] | None:
    """استخراج بيانات كتاب واحد بأمان، مع التعامل مع عناصر مفقودة.

    Args:
        book_element: عنصر BeautifulSoup يمثّل بطاقة كتاب واحدة.

    Returns:
        قاموس ببيانات الكتاب، أو None إذا كانت البيانات الأساسية مفقودة.
    """
    title_tag = book_element.select_one("h3 a")
    price_tag = book_element.select_one("p.price_color")

    if title_tag is None or price_tag is None:
        return None  # تخطَّ هذا العنصر بدلاً من رفع خطأ يوقف البرنامج كله

    return {
        "title": title_tag.get("title", "غير معروف"),
        "price": price_tag.get_text(strip=True),
    }

> **💡 نصيحة خبير:** لاحظ النمط هنا: بدلاً من ترك البرنامج ينهار عند أول خطأ (كصفحة مفقودة أو عنصر HTML غير متوقع)، نتعامل مع كل حالة فشل بلطف ونُكمل العمل. هذا مبدأ أساسي في الأتمتة الاحترافية: **السكربت الذي يعمل طوال الليل بدون إشراف بشري يجب أن يتحمّل الأخطاء الجزئية دون توقف كامل**.

## 21.10 ماذا عن المواقع التي تعتمد على JavaScript؟

مكتبة `requests` تجلب HTML **الخام** فقط كما يُرسله الخادم، دون تنفيذ أي JavaScript. لكن كثيرًا من المواقع الحديثة (تطبيقات React, Vue...) تبني محتواها **ديناميكيًا بعد تحميل الصفحة** عبر JavaScript، فتجد `requests` يُرجع صفحة شبه فارغة.

> **🔄 مقارنة: الأدوات حسب نوع الموقع**
>
> | نوع الموقع | الأداة المناسبة | السبب |
> |---|---|---|
> | HTML ثابت (المحتوى موجود في المصدر مباشرة) | `requests` + `BeautifulSoup` | سريع وخفيف، لا حاجة لمتصفح حقيقي |
> | يعتمد على JavaScript (المحتوى يُبنى ديناميكيًا) | **Selenium** أو **Playwright** | يُشغّلان متصفحًا حقيقيًا (أو بدون واجهة) وينتظران تحميل JavaScript قبل الاستخراج |
>
> **كيف تعرف أيّهما تحتاج؟** افتح الصفحة في المتصفح، ثم اضغط كليك يمين → «عرض مصدر الصفحة» (View Page Source، وليس Inspect). إذا رأيت البيانات التي تريدها موجودة في المصدر، فـ `requests` كافية. إذا كان المصدر شبه فارغ وكل المحتوى يظهر فقط بعد التحميل، تحتاج Selenium/Playwright.
>
> هذا الفصل يركّز على `requests`/`BeautifulSoup` لأنهما الأساس الذي يجب إتقانه أولاً، وهما كافيان لغالبية الحالات العملية والتدريبية.

## 21.11 مشروع تطبيقي متكامل: مستخرج بيانات كتب شامل

سنبني أداة أتمتة كاملة تستخرج بيانات كل الكتب من موقع `books.toscrape.com` عبر جميع صفحاته، تحفظها بشكل منظّم، ثم تنتج تحليلًا سريعًا عنها.

### هيكلية المشروع

```
book_scraper/
├── pyproject.toml
├── output/
└── src/
    └── book_scraper/
        ├── __init__.py
        ├── fetcher.py
        ├── parser.py
        ├── storage.py
        └── main.py
```

```bash
uv init book_scraper
cd book_scraper
uv add requests beautifulsoup4 lxml pandas
mkdir -p src/book_scraper output
touch src/book_scraper/__init__.py
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

### `src/book_scraper/fetcher.py` — جلب الصفحات بصلابة

```python
"""وحدة جلب صفحات الويب مع معالجة الأخطاء وإعادة المحاولة."""

import time

import requests

USER_AGENT = "Educational Book Scraper (learning purposes, github.com/example)"
REQUEST_TIMEOUT = 10
DELAY_BETWEEN_REQUESTS = 1.0  # ثانية -- احترام الخادم

def fetch_page(url: str, retries: int = 3) -> str | None:
    """جلب محتوى HTML لصفحة، مع إعادة محاولة عند الفشل.

    Args:
        url: رابط الصفحة المطلوبة.
        retries: عدد محاولات إعادة الطلب عند حدوث خطأ مؤقت.

    Returns:
        محتوى HTML كنص، أو None إذا فشلت كل المحاولات أو الصفحة غير موجودة.
    """
    headers = {"User-Agent": USER_AGENT}

    for attempt in range(1, retries + 1):
        try:
            response = requests.get(url, headers=headers, timeout=REQUEST_TIMEOUT)

            if response.status_code == 404:
                return None  # نهاية الصفحات -- ليس خطأ، بل إشارة توقف طبيعية

            response.raise_for_status()
            return response.text

        except requests.Timeout:
            print(f"⏱️ [محاولة {attempt}/{retries}] انتهت المهلة الزمنية: {url}")
        except requests.ConnectionError:
            print(f"🔌 [محاولة {attempt}/{retries}] تعذّر الاتصال: {url}")
        except requests.HTTPError as e:
            print(f"❌ [محاولة {attempt}/{retries}] خطأ HTTP: {e}")

        if attempt < retries:
            time.sleep(2 * attempt)

    print(f"🚫 فشلت كل المحاولات لجلب: {url}")
    return None

def polite_delay() -> None:
    """تأخير مهذّب بين الطلبات المتتالية لاحترام خادم الموقع."""
    time.sleep(DELAY_BETWEEN_REQUESTS)
```

🧾 **للقراءة فقط** — محتوى ملف داخل المشروع؛ يُكتب على القرص ثم يُشغَّل من الطرفية.

### `src/book_scraper/parser.py` — تحليل HTML واستخراج البيانات

```python
"""وحدة تحليل صفحات الكتب واستخراج البيانات منها."""

from dataclasses import dataclass

from bs4 import BeautifulSoup

RATING_WORDS_TO_NUMBERS = {
    "One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5,
}

@dataclass
class Book:
    """يمثّل بيانات كتاب واحد مُستخرجة من الموقع."""

    title: str
    price: float
    rating: int
    availability: str

def _parse_price(price_text: str) -> float:
    """تحويل نص السعر مثل '£51.77' إلى رقم عشري."""
    cleaned = price_text.replace("£", "").strip()
    return float(cleaned)

def _parse_rating(star_rating_tag) -> int:
    """استخراج رقم التقييم من اسم الـ class (مثل 'Three' -> 3)."""
    classes = star_rating_tag.get("class", [])
    for cls in classes:
        if cls in RATING_WORDS_TO_NUMBERS:
            return RATING_WORDS_TO_NUMBERS[cls]
    return 0

def extract_books_from_page(html: str) -> list[Book]:
    """استخراج كل الكتب من صفحة HTML واحدة.

    Args:
        html: محتوى HTML الخام للصفحة.

    Returns:
        قائمة من كائنات Book، مع تخطي أي عنصر بيانات ناقصة.
    """
    soup = BeautifulSoup(html, "lxml")
    book_elements = soup.select("article.product_pod")

    books: list[Book] = []

    for element in book_elements:
        title_tag = element.select_one("h3 a")
        price_tag = element.select_one("p.price_color")
        rating_tag = element.select_one("p.star-rating")
        availability_tag = element.select_one("p.instock.availability")

        if not all([title_tag, price_tag, rating_tag, availability_tag]):
            continue  # تخطَّ أي كتاب ببيانات ناقصة بدلاً من إيقاف البرنامج

        book = Book(
            title=title_tag.get("title", "غير معروف"),
            price=_parse_price(price_tag.get_text(strip=True)),
            rating=_parse_rating(rating_tag),
            availability=availability_tag.get_text(strip=True),
        )
        books.append(book)

    return books

def has_next_page(html: str) -> bool:
    """التحقق من وجود صفحة تالية في نظام التصفّح."""
    soup = BeautifulSoup(html, "lxml")
    return soup.select_one("li.next") is not None
```

🧾 **للقراءة فقط** — محتوى ملف داخل المشروع؛ يُكتب على القرص ثم يُشغَّل من الطرفية.

> **💡 نصيحة خبير:** استخدمنا `dataclass` (كما تعلمنا في الفصل 14) لتمثيل `Book` بدلاً من `dict` عادي. هذا يمنحنا Type Hints واضحة، اكتشاف أخطاء مبكر عبر mypy، وكودًا أسهل قراءة من `book["price"]` إلى `book.price`.

### `src/book_scraper/storage.py` — حفظ البيانات

```python
"""وحدة حفظ بيانات الكتب المُستخرجة."""

from dataclasses import asdict
from pathlib import Path

import pandas as pd

from book_scraper.parser import Book

def save_books_to_csv(books: list[Book], file_path: str | Path) -> None:
    """حفظ قائمة الكتب في ملف CSV باستخدام pandas.

    Args:
        books: قائمة كائنات Book المُستخرجة.
        file_path: مسار ملف الحفظ.
    """
    records = [asdict(book) for book in books]
    df = pd.DataFrame(records)
    df.to_csv(file_path, index=False, encoding="utf-8-sig")
```

🧾 **للقراءة فقط** — محتوى ملف داخل المشروع؛ يُكتب على القرص ثم يُشغَّل من الطرفية.

### `src/book_scraper/main.py` — تجميع كل شيء

```python
"""نقطة الدخول الرئيسية لأداة استخراج بيانات الكتب."""

from pathlib import Path

import pandas as pd

from book_scraper.fetcher import fetch_page, polite_delay
from book_scraper.parser import Book, extract_books_from_page, has_next_page
from book_scraper.storage import save_books_to_csv

BASE_URL = "https://books.toscrape.com/catalogue/page-{}.html"
OUTPUT_FILE = Path("output/books.csv")
MAX_PAGES = 5  # نحدد سقفًا احترازيًا لتجنب حلقة لا نهائية في حال خطأ غير متوقع

def scrape_all_books() -> list[Book]:
    """استخراج كل الكتب من كل الصفحات المتاحة.

    Returns:
        قائمة شاملة بكل الكتب المُستخرجة من كل الصفحات.
    """
    all_books: list[Book] = []
    page_number = 1

    while page_number <= MAX_PAGES:
        url = BASE_URL.format(page_number)
        print(f"📄 جلب الصفحة {page_number}...")

        html = fetch_page(url)
        if html is None:
            print("🏁 لا مزيد من الصفحات أو حدث خطأ غير قابل للتعافي. توقف.")
            break

        books_in_page = extract_books_from_page(html)
        all_books.extend(books_in_page)
        print(f"   ✅ تم استخراج {len(books_in_page)} كتاب")

        if not has_next_page(html):
            print("🏁 تم الوصول لآخر صفحة.")
            break

        page_number += 1
        polite_delay()

    return all_books

def print_analysis(books: list[Book]) -> None:
    """طباعة تحليل سريع لبيانات الكتب المُستخرجة."""
    if not books:
        print("لا توجد بيانات للتحليل.")
        return

    df = pd.DataFrame([{"title": b.title, "price": b.price, "rating": b.rating} for b in books])

    print("\n" + "=" * 50)
    print(" " * 12 + "تحليل بيانات الكتب المُستخرجة")
    print("=" * 50)
    print(f"إجمالي الكتب: {len(df)}")
    print(f"متوسط السعر: £{df['price'].mean():.2f}")
    print(f"أغلى كتاب: £{df['price'].max():.2f}")
    print(f"أرخص كتاب: £{df['price'].min():.2f}")
    print(f"متوسط التقييم: {df['rating'].mean():.1f} / 5")
    print("=" * 50)

def main() -> None:
    """تشغيل عملية الاستخراج الكاملة."""
    print("🚀 بدء استخراج بيانات الكتب من books.toscrape.com\n")

    books = scrape_all_books()

    print(f"\n💾 حفظ {len(books)} كتاب في {OUTPUT_FILE}...")
    OUTPUT_FILE.parent.mkdir(exist_ok=True)
    save_books_to_csv(books, OUTPUT_FILE)

    print_analysis(books)
    print(f"\n✅ اكتمل! راجع الملف: {OUTPUT_FILE}")

if __name__ == "__main__":
    main()
```

🧾 **للقراءة فقط** — محتوى ملف داخل المشروع؛ يُكتب على القرص ثم يُشغَّل من الطرفية.

تشغيل المشروع:

```bash
uv run python -m book_scraper.main
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

الناتج المتوقع:

```
🚀 بدء استخراج بيانات الكتب من books.toscrape.com

📄 جلب الصفحة 1...
   ✅ تم استخراج 20 كتاب
📄 جلب الصفحة 2...
   ✅ تم استخراج 20 كتاب
...

💾 حفظ 100 كتاب في output/books.csv...

==================================================
            تحليل بيانات الكتب المُستخرجة
==================================================
إجمالي الكتب: 100
متوسط السعر: £36.47
أغلى كتاب: £59.99
أرخص كتاب: £10.00
متوسط التقييم: 2.9 / 5
==================================================

✅ اكتمل! راجع الملف: output/books.csv
```

<details>
<summary>✅ الحل الكامل للمشروع (جميع الملفات مجتمعة)</summary>

جميع ملفات `fetcher.py`، `parser.py`، `storage.py`، `main.py` كما وردت بالكامل أعلاه تُشكّل الحل النموذجي الكامل. تثبيت التبعيات:

```bash
uv add requests beautifulsoup4 lxml pandas
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

**نقاط تصميم مهمة في هذا الحل:**
1. **فصل المسؤوليات**: `fetcher.py` (الشبكة) ≠ `parser.py` (التحليل) ≠ `storage.py` (الحفظ) ≠ `main.py` (التنسيق).
2. **الصلابة (Robustness)**: كل دالة تتعامل بلطف مع الفشل الجزئي (عنصر مفقود، صفحة غير موجودة) بدلاً من إيقاف البرنامج بالكامل.
3. **احترام الخادم**: `polite_delay()` بين كل طلب وآخر، و`User-Agent` واضح يُعرّف عن طبيعة الزيارة.
4. **سقف أمان** (`MAX_PAGES`): يمنع حلقة لا نهائية محتملة إذا حدث خطأ منطقي غير متوقع في اكتشاف «آخر صفحة».
5. **ربط مباشر بالفصل 20**: استخدمنا pandas للتحليل الفوري للبيانات المُستخرجة — توضيحًا عمليًا لكيفية تكامل مسارات الكتاب المختلفة معًا.

</details>

## 21.12 تمارين

### 🟢 مستوى مبتدئ

1. استخدم `requests` لجلب صفحة `https://books.toscrape.com/` واطبع `status_code` و`len(response.text)`.
2. استخدم BeautifulSoup لاستخراج **نص** عنوان الصفحة (`<title>`) من نفس الصفحة.
3. استخرج كل روابط `<a>` الموجودة في الصفحة الرئيسية واطبع أول 5 منها.

<details>
<summary>✅ الحل المقترح</summary>

In [ ]:
import requests
from bs4 import BeautifulSoup

response = requests.get("https://books.toscrape.com/", timeout=10)
print(response.status_code)
print(len(response.text))

soup = BeautifulSoup(response.text, "lxml")
print(soup.title.get_text(strip=True))

links = soup.find_all("a", href=True)
for link in links[:5]:
    print(link["href"])

</details>

### 🟡 مستوى متوسط

4. عدّل مشروع الكتب ليستخرج أيضًا **فئة المنتج (category)** من رابط صفحة الكتاب (تحتاج زيارة صفحة كل كتاب على حدة — رابط صفحة الكتاب موجود في `h3 a` كخاصية `href`). احذر: هذا يتطلب طلب HTTP إضافي لكل كتاب، فلا تنسَ `polite_delay()`!
5. اكتب دالة `filter_books_by_rating(books, min_rating)` تُرجع فقط الكتب التي تقييمها أكبر من أو يساوي قيمة معينة.

<details>
<summary>✅ الحل المقترح</summary>

```python
# تمرين 5 -- سهل التطبيق مباشرة على كائنات Book
def filter_books_by_rating(books: list[Book], min_rating: int) -> list[Book]:
    """إرجاع الكتب التي تقييمها أكبر من أو يساوي الحد الأدنى المطلوب."""
    return [book for book in books if book.rating >= min_rating]

# مثال استخدام
high_rated = filter_books_by_rating(books, min_rating=4)
print(f"عدد الكتب ذات التقييم 4+: {len(high_rated)}")
```

🧾 **للقراءة فقط** — مثال مقتطف يعتمد على `Book` من مثال سابق.

(تمرين 4 يتطلب دمج زيارة صفحة فرعية لكل كتاب، مشابه تمامًا لنمط `fetch_page` الموجود في المشروع، مع استخلاص `href` من `title_tag` وزيارته، ثم البحث عن عنصر الفئة في شريط التنقل Breadcrumb الخاص بصفحة الكتاب.)

</details>

### 🔴 مستوى تحدي

6. حوّل مشروع استخراج الكتب إلى **نسخة غير متزامنة (async)** باستخدام `aiohttp` (كما تعلمنا في الفصل 17) لجلب عدة صفحات بالتوازي بدلاً من التسلسل، **مع الحفاظ على تأخير مناسب يحترم الخادم** (مثلاً عبر `asyncio.Semaphore` لتحديد عدد الطلبات المتزامنة القصوى بـ 3 طلبات فقط في نفس الوقت).

<details>
<summary>✅ الحل المقترح (تصور مبسّط)</summary>

In [ ]:
import asyncio

import aiohttp

SEMAPHORE_LIMIT = 3  # عدد الطلبات المتزامنة القصوى -- احترامًا للخادم

async def fetch_page_async(session: aiohttp.ClientSession, url: str, sem: asyncio.Semaphore) -> str | None:
    """جلب صفحة بشكل غير متزامن مع تحديد التزامن عبر Semaphore."""
    async with sem:
        try:
            async with session.get(url, timeout=aiohttp.ClientTimeout(total=10)) as response:
                if response.status == 404:
                    return None
                response.raise_for_status()
                await asyncio.sleep(1)  # تأخير مهذّب حتى في الوضع المتزامن
                return await response.text()
        except aiohttp.ClientError as e:
            print(f"خطأ: {e}")
            return None

async def scrape_pages_async(urls: list[str]) -> list[str | None]:
    """جلب عدة صفحات بالتوازي مع تحديد عدد الطلبات المتزامنة."""
    sem = asyncio.Semaphore(SEMAPHORE_LIMIT)
    async with aiohttp.ClientSession() as session:
        tasks = [fetch_page_async(session, url, sem) for url in urls]
        return await asyncio.gather(*tasks)

# urls = [BASE_URL.format(i) for i in range(1, 6)]
# results = asyncio.run(scrape_pages_async(urls))

**ملاحظة مهمة:** `asyncio.Semaphore(3)` يضمن ألا يُرسل البرنامج أكثر من 3 طلبات في نفس اللحظة، رغم كونه «متزامنًا» من حيث البنية — هذا توازن جيد بين السرعة واحترام الخادم.

</details>

## 21.13 اختبر فهمك (Quiz)

<details>
<summary>1. ما الفرق بين وظيفة `requests` ووظيفة `BeautifulSoup`؟</summary>

`requests` مسؤولة عن **جلب** محتوى الصفحة عبر إرسال طلب HTTP واستقبال الاستجابة (HTML خام كنص). `BeautifulSoup` مسؤولة عن **تحليل** هذا النص الخام وتحويله إلى هيكل يمكن البحث فيه واستخراج بيانات محددة منه بسهولة. الأولى تتعامل مع الشبكة، والثانية تتعامل مع تحليل البنية.
</details>

<details>
<summary>2. لماذا يجب دائمًا تحديد `timeout` في طلبات `requests`؟</summary>

لأنه بدون `timeout`، إذا توقف الخادم البعيد عن الاستجابة لأي سبب، سيبقى البرنامج معلّقًا إلى الأبد دون أي رسالة خطأ أو طريقة للتعافي تلقائيًا. `timeout` يضمن أن البرنامج يرفع استثناءً قابلًا للمعالجة بعد مدة معقولة.
</details>

<details>
<summary>3. لماذا نكتب `class_="product"` بدلاً من `class="product"` في BeautifulSoup؟</summary>

لأن `class` كلمة مفتاحية محجوزة في لغة Python نفسها (تُستخدم لتعريف الأصناف Classes)، فاستخدامها كاسم معامل مباشرة يُسبب خطأ صياغي. BeautifulSoup يستخدم `class_` (بشرطة سفلية) كحل بديل لتجنب هذا التعارض.
</details>

<details>
<summary>4. اذكر 3 قواعد أخلاقية يجب اتباعها قبل استخراج بيانات أي موقع.</summary>

أي ثلاث من: فحص `robots.txt`، قراءة شروط الاستخدام، إضافة تأخير بين الطلبات لعدم إثقال الخادم، عدم استخراج بيانات شخصية أو محمية بحقوق نشر، استخدام API رسمي إن وُجد بدلاً من Web Scraping، وإرسال `User-Agent` واضح يُعرّف عن الزيارة.
</details>

<details>
<summary>5. متى تحتاج Selenium أو Playwright بدلاً من requests + BeautifulSoup؟</summary>

عندما يعتمد الموقع على JavaScript لبناء محتواه ديناميكيًا بعد تحميل الصفحة الأولي (كما في تطبيقات React أو Vue)، بحيث لا يحتوي HTML الخام الذي يُرجعه `requests` على البيانات المطلوبة فعليًا. Selenium/Playwright يُشغّلان متصفحًا حقيقيًا (أو بدون واجهة) وينتظران تنفيذ JavaScript قبل استخراج المحتوى النهائي.
</details>

<details>
<summary>6. ما الفرق بين `find`/`find_all` و `select`/`select_one` في BeautifulSoup؟</summary>

كلاهما يُستخدم للبحث عن عناصر HTML، لكن `find`/`find_all` يستخدمان صياغة BeautifulSoup الخاصة (وسم + خصائص Python)، بينما `select`/`select_one` يستخدمان صياغة محددات CSS القياسية، وهي غالبًا أقصر وأقوى للتعبير عن علاقات معقدة بين العناصر (مثل "كل `a` داخل `h3` داخل `article.product_pod`").
</details>

## 21.14 ملخص الفصل (Cheat Sheet)

```python
# ===== requests: الطلبات الأساسية =====
import requests
response = requests.get(url, headers={"User-Agent": "..."}, timeout=10)
response.status_code         # 200, 404, 500...
response.text                 # محتوى HTML/نص خام
response.json()               # تحويل تلقائي من JSON
response.raise_for_status()   # يرفع استثناء عند 4xx/5xx

try:
    response = requests.get(url, timeout=10)
except requests.Timeout:
    ...
except requests.ConnectionError:
    ...

# ===== BeautifulSoup: التحليل =====
from bs4 import BeautifulSoup
soup = BeautifulSoup(response.text, "lxml")

soup.find("div", class_="price")       # أول عنصر مطابق
soup.find_all("article")               # كل العناصر المطابقة -> قائمة
soup.select("article.product_pod h3 a")  # محدد CSS -> قائمة
soup.select_one("title")               # محدد CSS -> أول عنصر

tag.text / tag.get_text(strip=True)    # النص الظاهر
tag.get("href")                        # خاصية HTML بأمان (None إن لم توجد)
tag["href"]                            # خاصية HTML (يرفع خطأ إن لم توجد)

# ===== الأتمتة الآمنة =====
import time
time.sleep(1)                          # تأخير بين الطلبات (إلزامي أخلاقيًا)
```

🧾 **للقراءة فقط** — مثال مقتطف يعتمد على `tag` من مثال سابق.

| المبدأ الأخلاقي | التطبيق العملي |
|---|---|
| احترام الخادم | `time.sleep()` بين الطلبات |
| الشفافية | `User-Agent` واضح |
| الأمان | `timeout` في كل طلب |
| الصلابة | `try/except` + إعادة محاولة |
| احترام القواعد | فحص `robots.txt` وشروط الاستخدام |

## 21.15 ماذا بعد؟

لقد تعلّمت للتو كيف تجعل Python «يتصفح» الإنترنت نيابة عنك، ويستخرج بيانات منظّمة من فوضى صفحات HTML — وربطت هذه المهارة مباشرة بما تعلمته في الفصل السابق عبر تحليل البيانات المُستخرجة بـ pandas. هذا هو بالضبط نمط العمل الحقيقي في المشاريع الاحترافية: جمع البيانات → تنظيفها → تحليلها.

في الفصل القادم، ننتقل إلى مسار مثير آخر: **نافذتنا الأولى على الذكاء الاصطناعي**. سنتعرّف على مكتبة `scikit-learn`، ونتعلّم كيف «يتعلّم» الحاسوب من البيانات لاتخاذ تنبؤات، عبر بناء أول نموذج تعلّم آلي بسيط خطوة بخطوة.

---

### الخطوة التالية

- [الفصل الثاني والعشرون: مسار الذكاء الاصطناعي — نافذة أولى (scikit-learn)](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/chapter-22.ipynb)
- [فهرس كل الدفاتر](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/00-index.ipynb)

*الفصل الحادي والعشرون — من كتاب «بايثون 3.14: من الصفر إلى الاحتراف».*